In [1]:
import pandas as pd
import json
import re
import time
from llama_cpp import Llama
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm

In [2]:
print("🧠 Đang nạp file GGUF vào VRAM siêu tốc...")
llm = Llama(
    model_path="qwen_3b_sarcasm_gguf/Qwen2.5-3B-Instruct.Q4_K_M.gguf", 
    n_gpu_layers=-1, 
    n_ctx=1024,
    verbose=False
)

system_prompt = "Bạn là một chuyên gia ngôn ngữ học. Nhiệm vụ của bạn là phân tích tính châm biếm (sarcasm) trong bình luận dựa trên ngữ cảnh được cung cấp. BẮT BUỘC trả về định dạng JSON."

🧠 Đang nạp file GGUF vào VRAM siêu tốc...


llama_context: n_ctx_seq (1024) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


In [3]:
df = pd.read_csv("test_data.csv", encoding="utf-8-sig")
y_true = df['sarcasm_label'].tolist()
y_pred = []

print(f"\n🚀 Bắt đầu infer {len(df)} mẫu...")
start_time = time.time()

# THANH TIẾN ĐỘ TQDM Ở ĐÂY
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Tiến trình Test", unit="câu"):
    user_prompt = f"Input:\n- Context: {row['video_core_content']}\n- Comment: {row['comment']}"
    
    try:
        response = llm.create_chat_completion(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.0,
            response_format={"type": "json_object"}
        )
        
        raw_output = response['choices'][0]['message']['content']
        json_match = re.search(r'\{.*\}', raw_output, re.DOTALL)
        
        if json_match:
            data = json.loads(json_match.group())
            predicted_label = data['results'][0]['sarcasm_label']
        else:
            predicted_label = "Error"
            
    except Exception as e:
        predicted_label = "Error"

    y_pred.append(predicted_label)

# ==========================================
# 3. LÀM SẠCH & CHẤM ĐIỂM
# ==========================================
y_pred_clean = ["Non-Sarcastic" if p not in ["Sarcastic", "Non-Sarcastic"] else p for p in y_pred]
df['predicted_label'] = y_pred_clean
df.to_csv("test_result_3B.csv", index=False, encoding="utf-8-sig")

print("\n" + "="*50)
print("🏆 BÁO CÁO KẾT QUẢ MÔ HÌNH")
print("="*50)

f1_macro = f1_score(y_true, y_pred_clean, average='macro')
print(f"🔥 F1-MACRO SCORE: {f1_macro:.4f}\n")
print(classification_report(y_true, y_pred_clean, target_names=["Non-Sarcastic", "Sarcastic"]))


🚀 Bắt đầu infer 788 mẫu...


Tiến trình Test: 100%|██████████| 788/788 [23:15<00:00,  1.77s/câu]


🏆 BÁO CÁO KẾT QUẢ MÔ HÌNH
🔥 F1-MACRO SCORE: 0.6482

               precision    recall  f1-score   support

Non-Sarcastic       0.96      0.94      0.95       740
    Sarcastic       0.30      0.42      0.35        48

     accuracy                           0.90       788
    macro avg       0.63      0.68      0.65       788
 weighted avg       0.92      0.90      0.91       788

